In [2]:
import pandas as pd
from pathlib import Path
from evisionary_common import (
    DATA_PATH_MASTER, MOLECULE_CONCEPTS, QUERY_PATTERNS, valid_text_series,
    contains_ci, count_unique_pmids, summarize_sources, safe_fold_change,
)
OUT_DIR = Path("/Users/sogand/Desktop/Article_Figures/keyB/Advanced_Audits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH_MASTER)

s_sample = valid_text_series(df["sample_name"])
s_disease = valid_text_series(df["disease"])
s_mol = valid_text_series(df["molecule_type"])

# ---- A: Plasma specificity ----
print("=== PLASMA SPECIFICITY ===")

m_broad = contains_ci(s_sample, QUERY_PATTERNS["Plasma"])
m_seminal = contains_ci(s_sample, r"\bseminal\s+plasma\b")
m_blood = contains_ci(s_sample, r"\bblood\s+plasma\b")
m_refined = m_broad & ~m_seminal

# Compile the results into a structured list of dictionaries
plasma = [
    {"Strategy": "Broad plasma", "Records": int(m_broad.sum()),
     "Unique_PMIDs": count_unique_pmids(df, m_broad),
     "Note": "Maximizes recall; may include non-blood plasma."},
    {"Strategy": "Broad minus seminal", "Records": int(m_refined.sum()),
     "Unique_PMIDs": count_unique_pmids(df, m_refined),
     "Note": "Illustrative restriction removing a known non-blood context."},
    {"Strategy": "Explicit blood plasma", "Records": int(m_blood.sum()),
     "Unique_PMIDs": count_unique_pmids(df, m_blood),
     "Note": "High specificity; under-recovers unmodified 'plasma'."},
]

pd.DataFrame(plasma).to_csv(OUT_DIR / "Table_Plasma_Specificity.csv", index=False)
print(pd.DataFrame(plasma).to_string(index=False))

# ---- B: Source concentration ----
print("\n=== SOURCE CONCENTRATION ===")

# Define query concepts and their corresponding boolean masks
conc_queries = [
    ("Protein", contains_ci(s_mol, QUERY_PATTERNS["Protein"])),
    ("miRNA", contains_ci(s_mol, QUERY_PATTERNS["miRNA"])),       
    ("Lipid", contains_ci(s_mol, QUERY_PATTERNS["Lipid"])),       
    ("Plasma", contains_ci(s_sample, QUERY_PATTERNS["Plasma"])),
    ("Breast Cancer", contains_ci(s_disease, QUERY_PATTERNS["Breast Cancer"])),
]

rows = []
# Iterate over each concept to calculate source concentration metrics
for concept, mask in conc_queries:
    sub = df[mask].copy()
    sub["_src"] = valid_text_series(sub["source"])
    sub = sub[sub["_src"] != ""]
    
    total = len(sub)
    if total == 0:
        continue
        
    total_pmids = count_unique_pmids(df, mask)
    
    # Calculate representation metrics for each unique source
    for src, cnt in sub["_src"].value_counts().items():
        src_mask = mask & (valid_text_series(df["source"]) == src)
        rows.append({
            "Query": concept, 
            "Source": src, 
            "Records": int(cnt),
            "Pct_of_Query": round(cnt / total * 100, 2),
            "Source_PMIDs": count_unique_pmids(df, src_mask),
            "Total_Query_Records": total, 
            "Total_Query_PMIDs": total_pmids,
        })

# Save the source concentration table to CSV and print the output
pd.DataFrame(rows).to_csv(OUT_DIR / "Table_Source_Concentration.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))
print("\nAdvanced audits complete.")

=== PLASMA SPECIFICITY ===
             Strategy  Records  Unique_PMIDs                                                         Note
         Broad plasma      547           444              Maximizes recall; may include non-blood plasma.
  Broad minus seminal      544           442 Illustrative restriction removing a known non-blood context.
Explicit blood plasma      541           440        High specificity; under-recovers unmodified 'plasma'.

=== SOURCE CONCENTRATION ===
        Query       Source  Records  Pct_of_Query  Source_PMIDs  Total_Query_Records  Total_Query_PMIDs
      Protein Vesiclepedia   160806         77.45           548               207623                569
      Protein     ExoCarta    46817         22.55           203               207623                569
        miRNA Vesiclepedia    10091         62.56           116                16131                120
        miRNA     ExoCarta     6040         37.44            45                16131                120